# Solving ISR Mission Planning with ORBIT

This notebook runs the small (seed 42) scenario end to end through the **`orbit_hybrid`** solver
— ORBIT's simulator sampling a geography-aware assignment QUBO (`assignment_geographic_v2`),
followed by deterministic feasible-neighbourhood local search and nearest-neighbour sequencing —
and reports its performance:

1. Scenario and unsolved-mission overview
2. `orbit_hybrid` solve, validation, and a mission map / Gantt chart
3. Solution quality vs. an independently exact-solved ground state (SciPy/HiGHS MILP)
4. Repeated raw-ORBIT sampling to characterise ground-state hit rate and energy gap
5. `orbit_hybrid` vs. the classical `greedy` baseline

The QUBO covers task assignment only; sequencing and timing are classical post-processing, and no
optimality claim is made for the complete mission schedule. ORBIT SDK 0.0.2 does not expose its
simulator random seed, so runs are not bit-for-bit reproducible (`seed_applied: false`).

In [ ]:
from pathlib import Path

from IPython.display import Image

from isr_mission_planning.benchmark.energy import compute_qubo_energy
from isr_mission_planning.benchmark.ground_state import solve_qubo_exact
from isr_mission_planning.benchmark.orbit_sampling import run_orbit_sampling
from isr_mission_planning.benchmark.sample_analysis import summarise_samples
from isr_mission_planning.evaluation import validate_plan
from isr_mission_planning.generation import generate_scenario
from isr_mission_planning.solvers import GreedySolver, OrbitHybridSolver, SolverConfig
from isr_mission_planning.solvers.orbit.encoding_v2 import GeographicAssignmentV2Encoding, GeographicV2Config
from isr_mission_planning.visualisation import plot_gantt, plot_mission_map
from isr_mission_planning.visualisation.mission_map import plot_scenario

FIGURE_DIR = Path("../outputs/figures")
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

scenario = generate_scenario("large", seed=42)
print(f"{scenario.scenario_id}: {len(scenario.tasks)} tasks, {len(scenario.uavs)} UAVs, {len(scenario.bases)} bases")
plot_scenario(scenario, FIGURE_DIR / "unsolved_mission.png")
Image(str(FIGURE_DIR / "unsolved_mission.png"))

## 1. Solve with `orbit_hybrid`

Runs the real local ORBIT SDK simulator (`SimulatorSolver`) against the V2 assignment QUBO, then refines the sample with deterministic feasible-swap local search before decoding a full mission plan.

In [ ]:
config = SolverConfig(seed=42, orbit_full_sweeps=500, orbit_replicas=4, orbit_processes=1)
orbit_result = OrbitHybridSolver().solve(scenario, config)
orbit_plan = orbit_result.mission_plan
orbit_report = validate_plan(scenario, orbit_plan)

print(f"status:      {orbit_result.status.value}")
print(f"feasible:    {orbit_report.feasible}")
print(f"makespan:    {orbit_report.metrics.makespan:.2f}")
print(f"runtime:     {orbit_result.runtime_seconds:.3f}s")
print(f"conflicts:   {orbit_report.metrics.number_of_conflicts} (spatial deconfliction is a post-hoc warning, not a QUBO constraint)")
print()
print("time breakdown:")
for key in ("encoding_time", "orbit_execution_time", "local_search_time", "decoding_time"):
    print(f"  {key:22s} {orbit_result.solver_metadata[key]:.3f}s")
print()
print(f"raw ORBIT energy:      {orbit_result.solver_metadata['raw_orbit_energy']:.4f}")
print(f"refined energy:        {orbit_result.solver_metadata['refined_energy']:.4f}")
print(f"local-search moves:    {orbit_result.solver_metadata['local_moves']}")
print(f"assignment repairs:    {orbit_plan.metadata['assignment_repairs']} (QUBO constraint violations fixed during decoding)")

In [ ]:
plot_mission_map(scenario, orbit_plan, FIGURE_DIR / "orbit_hybrid_mission_map.png")
Image(str(FIGURE_DIR / "orbit_hybrid_mission_map.png"))

In [ ]:
plot_gantt(orbit_plan, FIGURE_DIR / "orbit_hybrid_gantt.png")
Image(str(FIGURE_DIR / "orbit_hybrid_gantt.png"))

## 2. `orbit_hybrid` vs. the classical `greedy` baseline

Same scenario, same validator, same metrics — greedy is a fast deterministic reference point, not a competing optimiser.

In [ ]:
greedy_result = GreedySolver().solve(scenario, config)
greedy_plan = greedy_result.mission_plan
greedy_report = validate_plan(scenario, greedy_plan)

rows = [
    ("greedy", greedy_report.feasible, greedy_report.metrics.makespan, greedy_report.metrics.number_of_conflicts, greedy_result.runtime_seconds),
    ("orbit_hybrid", orbit_report.feasible, orbit_report.metrics.makespan, orbit_report.metrics.number_of_conflicts, orbit_result.runtime_seconds),
]
header = f"{'solver':<14}{'feasible':<10}{'makespan':<12}{'conflicts':<11}{'runtime (s)':<12}"
print(header)
print("-" * len(header))
for name, feasible, makespan, conflicts, runtime in rows:
    print(f"{name:<14}{str(feasible):<10}{makespan:<12.2f}{conflicts:<11}{runtime:<12.3f}")

## 3. Solution quality vs. the exact ground state

Attempts to solve the same QUBO to a proven optimum with SciPy's HiGHS-backed MILP solver (binary linearisation), with a 15-minute time limit. If the limit is reached, the result is reported only as the best MILP incumbent—not as a ground state. This checks assignment-QUBO energy only; it is not a claim about the optimal complete mission schedule (sequencing/timing are classical).

In [ ]:
v2_config = GeographicV2Config(assignment_penalty=12.0, base_weight=0.75, cluster_weight=0.25, workload_weight=1.5)
encoding = GeographicAssignmentV2Encoding(v2_config).encode(scenario)
EXACT_TIME_LIMIT_SECONDS = 15 * 60
exact = solve_qubo_exact(encoding, time_limit_seconds=EXACT_TIME_LIMIT_SECONDS)

refined_state = orbit_result.solver_metadata["refined_state"]
refined_energy_check = compute_qubo_energy(refined_state, encoding)
gap = None if exact.ground_state_energy is None else refined_energy_check - exact.ground_state_energy
ground_state_reached = exact.optimality_proven and gap is not None and abs(gap) <= 1e-8

reference_label = "exact ground-state energy" if exact.optimality_proven else "best MILP incumbent"
print(f"exact-solver status:       {exact.status}")
print(f"exact-solver runtime:      {exact.runtime_seconds:.1f}s")
print(f"optimality proven:         {exact.optimality_proven}")
print(f"{reference_label + ':':27s}{exact.ground_state_energy:.4f}" if exact.ground_state_energy is not None else f"{reference_label + ':':27s}none")
print(f"orbit_hybrid final energy: {refined_energy_check:.4f}")
print(f"absolute energy gap:       {gap:.4f}" if gap is not None else "absolute energy gap:       unavailable")
print(f"ground state reached:      {ground_state_reached}")
if not exact.optimality_proven:
    print(f"Ground-state comparison is inconclusive after the {EXACT_TIME_LIMIT_SECONDS / 60:.0f}-minute limit.")

## 4. Repeated raw-ORBIT sampling: how often does it hit the ground state?

Bypasses local-search refinement to characterise the raw simulator: best/mean/median energy across independent runs, the fraction of individual samples landing exactly on the ground state, and a 95% Wilson confidence interval on that hit rate.

In [ ]:
REPEATS = 5
SAMPLES_PER_RUN = 2000

runs = [run_orbit_sampling(encoding, SAMPLES_PER_RUN, repeat=i, config=config) for i in range(REPEATS)]
all_states = [state for run in runs for state in run.states]
summary = summarise_samples(all_states, encoding, exact.ground_state_energy)

print(f"runs x samples:                 {REPEATS} x {SAMPLES_PER_RUN} = {summary.number_of_samples}")
print(f"best energy across all samples:  {summary.best_energy:.4f}")
print(f"mean / median energy:            {summary.mean_energy:.4f} / {summary.median_energy:.4f}")
print(f"samples exactly at ground state:{summary.ground_state_samples} ({summary.ground_state_sample_fraction:.2%})")
print(f"95% CI on hit rate:               [{summary.confidence_interval_95[0]:.2%}, {summary.confidence_interval_95[1]:.2%}]")
print(f"unique states observed:          {summary.unique_states} ({summary.unique_ground_states_observed} at ground state)")
print(f"absolute / relative energy gap:  {summary.absolute_gap:.4f} / {summary.relative_gap:.4%}" if summary.relative_gap is not None else f"absolute energy gap: {summary.absolute_gap:.4f}")

## Summary

- `orbit_hybrid` produces a **feasible, independently validated** mission plan from the real local ORBIT
  SDK simulator on every run (assignment-constraint violations, if any, are repaired deterministically
  and counted).
- Its assignment-QUBO energy is compared against a **proven exact optimum**, not a heuristic reference.
- Repeated raw sampling quantifies how reliably the simulator alone reaches that optimum before local
  search is applied, with a confidence interval rather than a single anecdotal run.
- `greedy` is kept only as a cheap deterministic sanity baseline.

Caveats carried over from the source formulation: the QUBO models task assignment with a load-balancing
surrogate; sequencing and timing are deterministic nearest-neighbour post-processing; spatial
deconfliction is reported as a warning, not enforced as an optimisation constraint; and ORBIT SDK 0.0.2
does not expose its simulator seed, so `seed_applied` is `false` throughout.